# FlatCast · Phase 1: wet/dry masks next to the gauge record

Each clear satellite image has been turned into a water / ambiguous / dry mask. This notebook checks the core idea behind FlatCast: **the lower the water at the Bird Island gauge, the more flat is exposed**, and wind explains much of what's left.

Run `uv run flatcast run --only gauges imagery masks pairs` first.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from flatcast.config import load_config
from flatcast.hydro import load_hourly
from flatcast.pairing import load_pairs
from flatcast.imagery.masks import load_stack
from flatcast import render

cfg = load_config('../config.yaml')
hourly = load_hourly(cfg)
pairs = load_pairs(cfg)
pairs = pairs[pairs['valid']].copy()
pairs['time'] = pd.to_datetime(pairs['time'], utc=True)
print(f"{len(pairs)} clear images paired with the gauge, {pairs.time.min():%Y-%m-%d} to {pairs.time.max():%Y-%m-%d}")
pairs[['time', 'sensor', 'clear_frac', 'h', 'speed', 'frac_dry', 'frac_water', 'frac_ambiguous']].tail()

## The gauge record, with every satellite image on it
Dots are coloured by how much of the area the satellite saw as dry.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
daily = hourly['h'].resample('D').mean()
ax.plot(daily.index, daily, color='#9aa7ad', lw=0.7, label='Bird Island daily mean')
sc = ax.scatter(pairs['time'], pairs['h'], c=pairs['frac_dry'], cmap=render.EXPOSURE_CMAP, s=22, edgecolor='k', linewidth=0.3, zorder=3)
fig.colorbar(sc, ax=ax, label='Share of area dry')
ax.set_ylabel('Water level (m NAVD88)')
ax.set_title('Satellite overpasses on the Bird Island water-level record')
ax.legend(frameon=False)
plt.show()

## Dry area against water level
If FlatCast's premise holds, this is a tight, falling curve. Colour shows the along-lagoon wind (north-south component): wind tilts the water, which spreads the points.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
sc = ax.scatter(pairs['h'], pairs['frac_dry'], c=pairs['v24'], cmap='coolwarm', vmin=-10, vmax=10, s=24, edgecolor='k', linewidth=0.3)
fig.colorbar(sc, ax=ax, label='24-h mean wind toward north (m/s)')
ax.set_xlabel('Gauge water level at overpass (m NAVD88)')
ax.set_ylabel('Share of area classified dry')
r = np.corrcoef(pairs['h'], pairs['frac_dry'])[0, 1]
ax.set_title(f'Lower water, more exposed flat (r = {r:.2f})')
plt.show()

## Masks across the range of water levels
Eight clear images from the lowest to the highest water.

In [ ]:
clear = pairs[pairs['clear_frac'] > 0.85].sort_values('h')
pick = clear.iloc[np.linspace(0, len(clear) - 1, 8).round().astype(int)]
masks = load_stack(cfg, list(pick.index))
fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
for ax, m, (_, row) in zip(axes.flat, masks, pick.iterrows()):
    ax.imshow(render.mask_rgb(m), interpolation='nearest')
    ax.set_title(f"{row['time'].tz_convert('America/Chicago'):%d %b %Y}\ngauge {row['h']:+.2f} m · wind {row['speed']:.0f} m/s", fontsize=10)
    render.axes_off(ax)
fig.suptitle('Sand = dry, teal = water, grey-green = ambiguous, light grey = cloud')
plt.tight_layout(); plt.show()

## How often is each pixel dry?
The flats that matter are the pixels that are neither always wet nor always dry.

In [ ]:
allm = load_stack(cfg, list(pairs.index))
valid = (allm == 0) | (allm == 1)
freq = np.where(valid.sum(0) > 10, (allm == 0).sum(0) / np.maximum(valid.sum(0), 1), np.nan)
fig, ax = plt.subplots(figsize=(8, 7.5))
im = ax.imshow(freq, cmap=render.EXPOSURE_CMAP, vmin=0, vmax=1)
fig.colorbar(im, ax=ax, label='Share of clear images in which the pixel was dry')
render.axes_off(ax); ax.set_title('Exposure frequency, all years'); plt.show()
print(f"Intermittently exposed (dry 5-95% of the time): {np.nansum((freq > 0.05) & (freq < 0.95)) * 0.0009:.1f} km2")